# Hypercare Agent Evaluation

Metrics: **routing accuracy**, **response quality**, **source relevance**, **escalation rate**, **observed errors**.

12 test cases covering all ticket categories + free-form questions.

In [ ]:
%pip install langchain-core langchain-openai langgraph pydantic --quiet

In [ ]:
import importlib
import os
import sys
from pathlib import Path

# Locate the app/ folder (works from the repo root, from notebooks/, or in a Snowflake workspace)
for candidate in [Path.cwd() / "app", Path.cwd().parent / "app", Path.cwd()]:
    if (candidate / "hypercare_agents.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    for root_dir, _, files in os.walk(Path.cwd().parent):
        if "hypercare_agents.py" in files:
            sys.path.insert(0, root_dir)
            break

import hypercare_agents
importlib.reload(hypercare_agents)
print(f"Loaded hypercare_agents from: {hypercare_agents.__file__}")

graph = hypercare_agents.build_graph()
print("Graph built OK")

In [ ]:
# 12 test scenarios covering all categories
TEST_CASES = [
    # 1. RULE_QUESTION (knowledge)
    {"id": "T01", "prompt": "How does SpotOn pricing work?",
     "expected_route": "knowledge", "expected_category": "RULE_QUESTION",
     "must_cite": ["DOC-"], "should_not": ["BK-", "QT-"]},

    # 2. RULE_QUESTION (knowledge) — SOC rules
    {"id": "T02", "prompt": "What rule applies for SOC empty containers on SpotOn?",
     "expected_route": "knowledge", "expected_category": "RULE_QUESTION",
     "must_cite": ["DOC-"]},

    # 3. BOOKING_ISSUE (data) — simple lookup
    {"id": "T03", "prompt": "Can you look up booking BK-001 and tell me its current status?",
     "expected_route": "data", "expected_category": "BOOKING_ISSUE",
     "must_cite": ["DATA:"],
     "check_fn": lambda r: "CONFIRMED" in r.get("answer", "")},

    # 4. BOOKING_ISSUE (data) — informal ID
    {"id": "T04", "prompt": "what about booking 3",
     "expected_route": "data", "expected_category": "BOOKING_ISSUE",
     "must_cite": ["DATA:"],
     "check_fn": lambda r: "BK-003" in r.get("answer", "") or "CONFIRMED" in r.get("answer", "")},

    # 5. PRICE_MISMATCH (data)
    {"id": "T05", "prompt": "There is a price mismatch on quote QT-009 for Shanghai to Le Havre. The customer was quoted a different rate than the price line.",
     "expected_route": "data", "expected_category": "PRICE_MISMATCH",
     "must_cite": ["DATA:"]},

    # 6. NO_OFFER (knowledge/both)
    {"id": "T06", "prompt": "A customer on route Le Havre to Shanghai says there is no SpotOn offer available. Why would there be no offer on this route?",
     "expected_route": ["knowledge", "both"], "expected_category": "NO_OFFER",
     "must_cite": ["DOC-"]},

    # 7. PRICE_LINE_INQUIRY (data)
    {"id": "T07", "prompt": "Can you check price line LN-005 and tell me its current rates and status?",
     "expected_route": "data", "expected_category": "PRICE_LINE_INQUIRY",
     "must_cite": ["DATA:"]},

    # 8. TICKET_INQUIRY (data)
    {"id": "T08", "prompt": "What's the status of ticket TK-003?",
     "expected_route": "data", "expected_category": "TICKET_INQUIRY",
     "must_cite": ["DATA:"]},

    # 9. FUNCTIONALITY / RULE_QUESTION (knowledge)
    {"id": "T09", "prompt": "How does the charge auto flag work on price lines? What's the difference with line flexibility?",
     "expected_route": "knowledge",
     "expected_category": ["FUNCTIONALITY", "RULE_QUESTION"],
     "must_cite": ["DOC-"]},

    # 10. HAZ surcharge — PRICE_MISMATCH (data)
    {"id": "T10", "prompt": "Quote QT-012 for a hazardous booking — the HAZ surcharge doesn't seem applied. Can you check?",
     "expected_route": ["data", "both"], "expected_category": "PRICE_MISMATCH",
     "must_cite": ["DATA:"]},

    # 11. Ambiguous → clarify
    {"id": "T11", "prompt": "Something is wrong with the rate",
     "expected_route": "clarify", "expected_category": ["UNKNOWN", "PRICE_MISMATCH"],
     "must_cite": [],
     "check_fn": lambda r: "?" in r.get("answer", "")},

    # 12. Contradictory → escalate or data
    {"id": "T12", "prompt": "The system shows two completely different rates for the same container type on price line LN-001. One screen shows 1200 and another shows 1500. Multiple customers have complained.",
     "expected_route": ["data", "escalate", "both"],
     "expected_category": "PRICE_MISMATCH",
     "must_cite": ["DATA:"]},
]

print(f"{len(TEST_CASES)} test cases defined")

In [ ]:
# Run all test cases through the agent pipeline
results = []

for i, tc in enumerate(TEST_CASES):
    tid = tc["id"]
    print(f"Running {tid}: {tc['prompt'][:60]}...")
    start = time.time()
    try:
        result = hypercare_agents.process_ticket(
            graph, subject=tc["prompt"][:120], body=tc["prompt"])
        elapsed = round(time.time() - start, 1)
        answer = result.get("answer", "")
        trace = result.get("trace", [])

        # Extract route and category from triage trace
        actual_route, actual_category = "", ""
        for step in trace:
            if step["node"] == "triage":
                for part in step["detail"].split():
                    if part.startswith("route="): actual_route = part.split("=", 1)[1]
                    if part.startswith("category="): actual_category = part.split("=", 1)[1]

        # Check routing
        exp_r = tc["expected_route"]
        route_ok = actual_route in exp_r if isinstance(exp_r, list) else actual_route == exp_r

        # Check category
        exp_c = tc["expected_category"]
        cat_ok = actual_category in exp_c if isinstance(exp_c, list) else actual_category == exp_c

        # Check citations
        cite_ok = all(c in answer for c in tc.get("must_cite", []))

        # Check hallucination guards
        no_halluc = not any(s in answer for s in tc.get("should_not", []))

        # Custom check
        custom_ok = tc["check_fn"](result) if tc.get("check_fn") else True

        # KB healing
        kb_proposal = result.get("kb_healing_proposal", "")
        kb_gap = False
        if kb_proposal:
            try: kb_gap = json.loads(kb_proposal).get("gap_detected", False)
            except: pass

        results.append({
            "id": tid, "prompt": tc["prompt"][:60],
            "expected_route": str(exp_r), "actual_route": actual_route, "route_ok": route_ok,
            "expected_category": str(exp_c), "actual_category": actual_category, "category_ok": cat_ok,
            "cite_ok": cite_ok, "no_hallucination": no_halluc, "custom_ok": custom_ok,
            "escalated": result.get("escalated", False), "kb_gap_detected": kb_gap,
            "elapsed_s": elapsed, "error": None,
        })
        status = "OK" if (route_ok and cat_ok and cite_ok and custom_ok) else "ISSUE"
        print(f"  -> {elapsed}s | route={actual_route} | category={actual_category} | {status}")

    except Exception as e:
        elapsed = round(time.time() - start, 1)
        results.append({
            "id": tid, "prompt": tc["prompt"][:60],
            "expected_route": str(tc["expected_route"]), "actual_route": "ERROR", "route_ok": False,
            "expected_category": str(tc["expected_category"]), "actual_category": "ERROR", "category_ok": False,
            "cite_ok": False, "no_hallucination": False, "custom_ok": False,
            "escalated": False, "kb_gap_detected": False, "elapsed_s": elapsed, "error": str(e),
        })
        print(f"  -> ERROR in {elapsed}s: {e}")

print(f"\nAll {len(results)} tests complete.")

In [ ]:
# Results table
import pandas as pd

df = pd.DataFrame(results)
df[["id", "prompt", "actual_route", "route_ok", "actual_category", "category_ok",
    "cite_ok", "no_hallucination", "custom_ok", "escalated", "kb_gap_detected", "elapsed_s", "error"]]

In [ ]:
# Metrics summary
n = len(df)
valid = df[df["error"].isna()]
n_valid = len(valid)

route_acc = valid["route_ok"].mean() * 100
cat_acc = valid["category_ok"].mean() * 100
cite_acc = valid["cite_ok"].mean() * 100
no_halluc = valid["no_hallucination"].mean() * 100
custom_acc = valid["custom_ok"].mean() * 100
esc_rate = valid["escalated"].mean() * 100
kb_rate = valid["kb_gap_detected"].mean() * 100
avg_lat = valid["elapsed_s"].mean()
err_count = n - n_valid

print("=" * 60)
print("         HYPERCARE AGENT EVALUATION RESULTS")
print("=" * 60)
print(f"  Test cases run:          {n}")
print(f"  Errors (crashes):        {err_count} / {n}")
print()
print(f"  Routing accuracy:        {route_acc:.0f}% ({int(valid['route_ok'].sum())}/{n_valid})")
print(f"  Category accuracy:       {cat_acc:.0f}% ({int(valid['category_ok'].sum())}/{n_valid})")
print(f"  Source citation:         {cite_acc:.0f}% ({int(valid['cite_ok'].sum())}/{n_valid})")
print(f"  No hallucination:        {no_halluc:.0f}% ({int(valid['no_hallucination'].sum())}/{n_valid})")
print(f"  Custom checks passed:    {custom_acc:.0f}% ({int(valid['custom_ok'].sum())}/{n_valid})")
print()
print(f"  Escalation rate:         {esc_rate:.0f}% ({int(valid['escalated'].sum())}/{n_valid})")
print(f"  KB gaps detected:        {kb_rate:.0f}% ({int(valid['kb_gap_detected'].sum())}/{n_valid})")
print(f"  Avg latency:             {avg_lat:.1f}s")
print("=" * 60)

# Flag failures
failures = valid[~(valid["route_ok"] & valid["category_ok"] & valid["cite_ok"] & valid["custom_ok"])]
if len(failures) > 0:
    print(f"\n{len(failures)} test(s) with issues:")
    for _, row in failures.iterrows():
        issues = []
        if not row["route_ok"]: issues.append(f"route: expected {row['expected_route']}, got {row['actual_route']}")
        if not row["category_ok"]: issues.append(f"category: expected {row['expected_category']}, got {row['actual_category']}")
        if not row["cite_ok"]: issues.append("missing expected citations")
        if not row["custom_ok"]: issues.append("custom check failed")
        print(f"  {row['id']}: {', '.join(issues)}")
else:
    print("\nAll tests passed.")

In [ ]:
# Visual dashboard
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = 'white'

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 1. Routing accuracy
rp = int(valid["route_ok"].sum()); rf = n_valid - rp
axes[0].pie([rp, rf], labels=[f"Correct\n{rp}", f"Wrong\n{rf}"],
            colors=["#4CAF50", "#f44336"], autopct="%1.0f%%", startangle=90)
axes[0].set_title("Routing Accuracy")

# 2. Source citation
cp = int(valid["cite_ok"].sum()); cf = n_valid - cp
axes[1].pie([cp, cf], labels=[f"Cited\n{cp}", f"Missing\n{cf}"],
            colors=["#2196F3", "#FF9800"], autopct="%1.0f%%", startangle=90)
axes[1].set_title("Source Relevance")

# 3. Escalation
ec = int(valid["escalated"].sum()); rc = n_valid - ec
axes[2].pie([rc, ec], labels=[f"Resolved\n{rc}", f"Escalated\n{ec}"],
            colors=["#4CAF50", "#FF9800"], autopct="%1.0f%%", startangle=90)
axes[2].set_title("Escalation Rate")

# 4. Latency
axes[3].bar(valid["id"], valid["elapsed_s"], color="#607D8B")
axes[3].set_ylabel("Seconds")
axes[3].set_title("Response Latency")
axes[3].tick_params(axis='x', rotation=45)
axes[3].axhline(y=avg_lat, color='red', linestyle='--', label=f'Avg: {avg_lat:.1f}s')
axes[3].legend()

plt.tight_layout()
plt.show()